# American Household Financial Health 2024

**Survey of Household Economics and Decisionmaking (SHED)**  
Federal Reserve Board | October 2024 | n=12,295

---

A three-act narrative exploring the state of household finances in America.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Figure settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [2]:
# Load the dataset
df = pd.read_csv('data_optimized.csv')

print(f"Dataset loaded: {len(df):,} respondents")
print(f"Field period: October 18-31, 2024")
print(f"Variables: {len(df.columns)}")

Dataset loaded: 12,295 respondents
Field period: October 18-31, 2024
Variables: 385


/var/folders/kn/k2gq2hds1tg0q3tdmc6sxh4m0000gn/T/ipykernel_10834/229530503.py:2: DtypeWarning: Columns (13,39,48,98,113,115,116,117,118,119,130,149,150,151,152,180,183,184,185,186,187,188,189,190,191,192,206,207,208,209,238,248,258,259,260,261,262,263,264,299,364,365,366,367) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data_optimized.csv')


---
---

# ACT 1: THE FOUNDATION

## *Where money comes from and where it must go*

Understanding household financial health begins with the basics: Do people earn enough? Are they working? Can they afford shelter? And critically—does income cover expenses?

## 1.1 Income Landscape

In [3]:
print("="*70)
print("INCOME LANDSCAPE")
print("="*70)

# Income distribution (simplified 4 categories)
print("\n--- Household Income Distribution (inc_4cat_50k) ---\n")
if 'inc_4cat_50k' in df.columns:
    inc_counts = df['inc_4cat_50k'].value_counts()
    inc_pct = df['inc_4cat_50k'].value_counts(normalize=True) * 100
    
    for category in ['Less than $25,000', '$25,000–$49,999', '$50,000–$99,999', '$100,000 or more']:
        if category in inc_counts.index:
            print(f"  {category:.<35} {inc_counts[category]:>6,} ({inc_pct[category]:>5.1f}%)")
    
    # Key threshold
    above_50k = inc_pct.get('$50,000–$99,999', 0) + inc_pct.get('$100,000 or more', 0)
    print(f"\n  → {above_50k:.1f}% earn $50,000 or more")

# Primary income sources
print("\n--- Primary Income Sources (I0 series) ---")
print("(Note: Households can have multiple income sources)\n")
income_sources = {
    'I0_a': 'Wages or salaries',
    'I0_c': 'Social Security',
    'I0_b': 'Self-employment',
    'I0_d': 'Pension/retirement account'
}

for var, label in income_sources.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = yes_count / total * 100
            print(f"  {label:.<35} {pct:>5.1f}%")

INCOME LANDSCAPE

--- Household Income Distribution (inc_4cat_50k) ---

  Less than $25,000..................  2,114 ( 17.2%)
  $25,000–$49,999....................  2,000 ( 16.3%)
  $50,000–$99,999....................  3,370 ( 27.4%)
  $100,000 or more...................  4,811 ( 39.1%)

  → 66.5% earn $50,000 or more

--- Primary Income Sources (I0 series) ---
(Note: Households can have multiple income sources)

  Wages or salaries..................  64.7%
  Social Security....................  32.2%
  Self-employment....................  38.8%
  Pension/retirement account.........   5.3%


In [ ]:
# Visualizations for Income Landscape

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Income Distribution (Horizontal Bar)
if 'inc_4cat_50k' in df.columns:
    inc_data = df['inc_4cat_50k'].value_counts()
    categories = ['Less than $25,000', '$25,000–$49,999', '$50,000–$99,999', '$100,000 or more']
    values = [inc_data.get(cat, 0) for cat in categories]
    colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
    
    ax1.barh(categories, values, color=colors, edgecolor='black', linewidth=1.2)
    ax1.set_xlabel('Number of Households', fontsize=12, fontweight='bold')
    ax1.set_title('Household Income Distribution', fontsize=13, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    total = sum(values)
    for i, (cat, val) in enumerate(zip(categories, values)):
        pct = val / total * 100
        ax1.text(val + 100, i, f'{pct:.1f}%', va='center', fontsize=10, fontweight='bold')

# Chart 2: Income Sources (Multiple sources possible)
income_sources = {
    'I0_a': 'Wages/Salaries',
    'I0_c': 'Social Security',
    'I0_b': 'Self-employment',
    'I0_d': 'Pension/Retirement'
}

source_pcts = []
source_labels = []
for var, label in income_sources.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        source_pcts.append(pct)
        source_labels.append(label)

ax2.barh(source_labels, source_pcts, color='#1f77b4', edgecolor='black', linewidth=1.2)
ax2.set_xlabel('% of Households', fontsize=12, fontweight='bold')
ax2.set_title('Primary Income Sources\n(Households can have multiple)', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, pct in enumerate(source_pcts):
    ax2.text(pct + 1, i, f'{pct:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 1.2 Employment Status

In [4]:
print("="*70)
print("EMPLOYMENT STATUS")
print("="*70)

# Work status
print("\n--- Work Status Last Month (D1A) ---\n")
if 'D1A' in df.columns:
    d1a_counts = df['D1A'].value_counts()
    d1a_pct = df['D1A'].value_counts(normalize=True) * 100
    
    for category in d1a_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {d1a_counts[category]:>6,} ({d1a_pct[category]:>5.1f}%)")
    
    # Labor force participation
    working = df['D1A'].isin(['Yes, full-time', 'Yes, part-time']).sum()
    total = df['D1A'].notna().sum()
    if total > 0:
        lfp = working / total * 100
        print(f"\n  → Labor Force Participation Rate: {lfp:.1f}%")
        
        # Full-time vs part-time split
        ft = (df['D1A'] == 'Yes, full-time').sum()
        pt = (df['D1A'] == 'Yes, part-time').sum()
        if working > 0:
            print(f"  → Of workers: {ft/working*100:.1f}% full-time, {pt/working*100:.1f}% part-time")

# Job instability indicator
print("\n--- Job Instability ---\n")
if 'D44_d' in df.columns:
    laid_off = (df['D44_d'] == 'Yes').sum()
    total_resp = df['D44_d'].notna().sum()
    if total_resp > 0:
        pct_laid_off = laid_off / total_resp * 100
        print(f"  Laid off or lost job in past year........ {pct_laid_off:.1f}%")

EMPLOYMENT STATUS

--- Work Status Last Month (D1A) ---

  Yes..........................................  7,066 ( 57.5%)
  No...........................................  5,229 ( 42.5%)

  → Labor Force Participation Rate: 0.0%

--- Job Instability ---

  Laid off or lost job in past year........ 12.2%


In [ ]:
# Visualization for Employment Status - Job Instability Focus

fig, ax = plt.subplots(figsize=(12, 6))

# Job transitions from D44 series
job_transitions = {
    'D44_a': 'Received a raise',
    'D44_b': 'Received a promotion',
    'D44_c': 'Took on a second job',
    'D44_e': 'Started a new job',
    'D44_f': 'Voluntarily left job',
    'D44_d': 'Laid off or lost job'
}

transition_pcts = []
transition_labels = []
colors_transitions = []

for var, label in job_transitions.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        transition_pcts.append(pct)
        transition_labels.append(label)
        
        # Color code by type
        if var in ['D44_a', 'D44_b']:  # Positive - raises, promotions
            colors_transitions.append('#2ca02c')
        elif var in ['D44_d', 'D44_f']:  # Negative - layoffs, leaving
            colors_transitions.append('#d62728')
        else:  # Neutral - new job, second job
            colors_transitions.append('#ff7f0e')

bars = ax.barh(transition_labels, transition_pcts, color=colors_transitions,
               edgecolor='black', linewidth=1.5, alpha=0.85)

ax.set_xlabel('Percentage of Respondents', fontsize=12, fontweight='bold')
ax.set_title('Job Transitions in Past Year', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add percentage labels
for bar, pct in zip(bars, transition_pcts):
    ax.text(pct + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 1.3 Housing Security

In [5]:
print("="*70)
print("HOUSING SECURITY")
print("="*70)

# Housing tenure
print("\n--- Housing Tenure (GH1) ---\n")
if 'GH1' in df.columns:
    gh1_counts = df['GH1'].value_counts()
    gh1_pct = df['GH1'].value_counts(normalize=True) * 100
    
    for category in gh1_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {gh1_counts[category]:>6,} ({gh1_pct[category]:>5.1f}%)")

# Housing costs
print("\n--- Monthly Housing Costs ---\n")

# Renters
if 'R3' in df.columns:
    rent_data = pd.to_numeric(df['R3'], errors='coerce')
    rent_valid = rent_data[rent_data.notna() & (rent_data > 0)]
    
    if len(rent_valid) > 0:
        print(f"  RENTERS (n={len(rent_valid):,}):")
        print(f"    Median monthly rent................ ${rent_valid.median():>8,.0f}")
        print(f"    Mean monthly rent.................. ${rent_valid.mean():>8,.0f}")

# Homeowners with mortgage
if 'M4' in df.columns:
    mortgage_data = pd.to_numeric(df['M4'], errors='coerce')
    mortgage_valid = mortgage_data[mortgage_data.notna() & (mortgage_data > 0)]
    
    if len(mortgage_valid) > 0:
        print(f"\n  HOMEOWNERS WITH MORTGAGE (n={len(mortgage_valid):,}):")
        print(f"    Median monthly mortgage............ ${mortgage_valid.median():>8,.0f}")
        print(f"    Mean monthly mortgage.............. ${mortgage_valid.mean():>8,.0f}")

HOUSING SECURITY

--- Housing Tenure (GH1) ---

  Own your home with a mortgage or loan........  5,078 ( 41.3%)
  Pay rent.....................................  3,206 ( 26.1%)
  Own your home free and clear (without a mortgage or loan)  3,106 ( 25.3%)
  Neither own nor pay rent.....................    905 (  7.4%)

--- Monthly Housing Costs ---

  RENTERS (n=3,185):
    Median monthly rent................ $   1,200
    Mean monthly rent.................. $   1,379

  HOMEOWNERS WITH MORTGAGE (n=4,977):
    Median monthly mortgage............ $   1,500
    Mean monthly mortgage.............. $   1,724


In [ ]:
# Visualization for Housing Security

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Housing Tenure (Pie Chart)
if 'GH1' in df.columns:
    gh1_data = df['GH1'].value_counts()
    
    labels = []
    values = []
    for category in gh1_data.index:
        if pd.notna(category):
            # Shorten labels for better display
            short_label = category.replace('Own your home ', '').replace('(', '\n(')
            labels.append(short_label)
            values.append(gh1_data[category])
    
    colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']
    
    ax1.pie(values, labels=labels, autopct='%1.1f%%', startangle=45,
            colors=colors, textprops={'fontsize': 9, 'fontweight': 'bold'})
    ax1.set_title('Housing Tenure', fontsize=13, fontweight='bold')

# Chart 2: Monthly Housing Costs (Box plot comparison)
housing_costs = []
housing_labels = []

if 'R3' in df.columns:
    rent_data = pd.to_numeric(df['R3'], errors='coerce')
    rent_valid = rent_data[rent_data.notna() & (rent_data > 0)]
    if len(rent_valid) > 0:
        housing_costs.append(rent_valid)
        housing_labels.append('Rent')

if 'M4' in df.columns:
    mortgage_data = pd.to_numeric(df['M4'], errors='coerce')
    mortgage_valid = mortgage_data[mortgage_data.notna() & (mortgage_data > 0)]
    if len(mortgage_valid) > 0:
        housing_costs.append(mortgage_valid)
        housing_labels.append('Mortgage')

if housing_costs:
    bp = ax2.boxplot(housing_costs, labels=housing_labels, patch_artist=True,
                     widths=0.5, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
    
    # Color the boxes
    colors_box = ['#ff7f0e', '#2ca02c']
    for patch, color in zip(bp['boxes'], colors_box[:len(bp['boxes'])]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax2.set_ylabel('Monthly Cost ($)', fontsize=12, fontweight='bold')
    ax2.set_title('Monthly Housing Costs\n(Box plots show median, quartiles, outliers)', 
                  fontsize=13, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

# Visualization for Housing Security - Housing Tenure

fig, ax = plt.subplots(figsize=(8, 8))

if 'GH1' in df.columns:
    gh1_data = df['GH1'].value_counts()
    
    labels = []
    values = []
    for category in gh1_data.index:
        if pd.notna(category):
            # Shorten labels for better display
            short_label = category.replace('Own your home ', '').replace('(', '\n(')
            labels.append(short_label)
            values.append(gh1_data[category])
    
    colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']
    
    ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=45,
            colors=colors, textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_title('Housing Tenure', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Housing Costs

fig, ax = plt.subplots(figsize=(10, 6))

housing_costs_data = []
housing_labels = []
housing_colors = []

# Calculate median rent
if 'R3' in df.columns:
    rent_data = pd.to_numeric(df['R3'], errors='coerce')
    rent_valid = rent_data[rent_data.notna() & (rent_data > 0)]
    if len(rent_valid) > 0:
        housing_costs_data.append(rent_valid.median())
        housing_labels.append('Median Monthly Rent')
        housing_colors.append('#ff7f0e')

# Calculate median mortgage
if 'M4' in df.columns:
    mortgage_data = pd.to_numeric(df['M4'], errors='coerce')
    mortgage_valid = mortgage_data[mortgage_data.notna() & (mortgage_data > 0)]
    if len(mortgage_valid) > 0:
        housing_costs_data.append(mortgage_valid.median())
        housing_labels.append('Median Monthly Mortgage')
        housing_colors.append('#2ca02c')

if housing_costs_data:
    bars = ax.bar(housing_labels, housing_costs_data, color=housing_colors,
                  edgecolor='black', linewidth=2, alpha=0.85, width=0.5)
    
    ax.set_ylabel('Monthly Cost (Dollars)', fontsize=12, fontweight='bold')
    ax.set_title('Monthly Housing Costs Comparison', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Add value labels
    for bar, val in zip(bars, housing_costs_data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'${val:,.0f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print("="*70)
print("THE CRITICAL GAP: INCOME VS. SPENDING")
print("="*70)

print("\n--- Spending Relative to Income (I20) ---\n")
if 'I20' in df.columns:
    i20_counts = df['I20'].value_counts()
    i20_pct = df['I20'].value_counts(normalize=True) * 100
    
    for category in i20_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {i20_counts[category]:>6,} ({i20_pct[category]:>5.1f}%)")
    
    # Key insight - at or above income
    same = i20_pct.get('The same as your income', 0)
    more = i20_pct.get('More than your income', 0)
    at_or_above = same + more
    
    print(f"\n  ⚠️  {at_or_above:.1f}% of households spend at or MORE than they earn")
    print(f"      ({same:.1f}% spend same as income + {more:.1f}% spend more)")

print("\n" + "="*70)
print("END OF ACT 1")
print("="*70)

---
---

# ACT 2: THE STRESS TEST

## *When crisis hits, what happens?*

Having established the foundation, we now examine resilience: Can households weather financial shocks? What are they sacrificing to stay afloat? Where are the breaking points?

# Visualization for Spending Relative to Income

fig, ax = plt.subplots(figsize=(10, 6))

if 'I20' in df.columns:
    i20_data = df['I20'].value_counts()
    
    categories = ['Less than your income', 'The same as your income', 'More than your income']
    values = [i20_data.get(cat, 0) for cat in categories]
    colors = ['#2ca02c', '#ff7f0e', '#d62728']
    
    # Calculate percentages
    total = sum(values)
    percentages = [val/total * 100 for val in values]
    
    bars = ax.bar(categories, percentages, color=colors, edgecolor='black', linewidth=2, alpha=0.8)
    
    ax.set_ylabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_title('Spending Relative to Income', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage labels
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for The Critical Gap: Income vs Spending

fig, ax = plt.subplots(figsize=(10, 6))

if 'I20' in df.columns:
    i20_data = df['I20'].value_counts()
    
    categories = ['Less than your income', 'The same as your income', 'More than your income']
    values = [i20_data.get(cat, 0) for cat in categories]
    colors = ['#2ca02c', '#ff7f0e', '#d62728']
    
    bars = ax.bar(categories, values, color=colors, edgecolor='black', linewidth=2, alpha=0.8)
    
    ax.set_ylabel('Number of Households', fontsize=12, fontweight='bold')
    ax.set_title('Spending Relative to Income\n⚠️ Nearly half spend at or above their income', 
                 fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value and percentage labels
    total = sum(values)
    for i, (bar, val) in enumerate(zip(bars, values)):
        height = bar.get_height()
        pct = val / total * 100
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(val):,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Highlight the warning zone
    same_idx = categories.index('The same as your income')
    more_idx = categories.index('More than your income')
    ax.axvspan(same_idx - 0.4, more_idx + 0.4, alpha=0.15, color='red', 
               label='At or Above Income (Financial Risk)')
    ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [7]:
print("="*70)
print("RESILIENCE CAPACITY")
print("="*70)

# The iconic $400 test
print("\n--- The $400 Emergency Test (pay_casheqv) ---\n")
if 'pay_casheqv' in df.columns:
    pay_cash = df['pay_casheqv'].value_counts()
    pay_cash_pct = df['pay_casheqv'].value_counts(normalize=True) * 100
    
    for category in pay_cash.index:
        if pd.notna(category):
            print(f"  {category:.<45} {pay_cash[category]:>6,} ({pay_cash_pct[category]:>5.1f}%)")
    
    can_handle = pay_cash_pct.get('Yes', 0)
    print(f"\n  💰 {can_handle:.1f}% can handle a $400 emergency with cash/equivalent")
    print(f"  ⚠️  {100-can_handle:.1f}% CANNOT handle a $400 emergency")

# Emergency savings depth
print("\n--- Emergency Savings: Months of Expenses Covered (EF2) ---\n")
if 'EF2' in df.columns:
    ef2_counts = df['EF2'].value_counts()
    ef2_pct = df['EF2'].value_counts(normalize=True) * 100
    
    for category in ef2_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {ef2_counts[category]:>6,} ({ef2_pct[category]:>5.1f}%)")

# Credit card behavior - focus on carrying debt
print("\n--- Credit Card Debt Behavior (C3P) ---\n")
if 'C3P' in df.columns:
    c3p_counts = df['C3P'].value_counts()
    c3p_pct = df['C3P'].value_counts(normalize=True) * 100
    
    for category in c3p_counts.index:
        if pd.notna(category):
            print(f"  {category:.<55} {c3p_pct[category]:>5.1f}%")
    
    # Those carrying balances (paid less than full)
    min_only = c3p_pct.get('did not pay or paid less than the minimum payment on at least one card', 0)
    paid_min = c3p_pct.get('paid at least the minimum payment on all credit cards', 0)
    
    print(f"\n  → {paid_min:.1f}% paid at least minimum (likely carrying balances)")
    print(f"  → {min_only:.1f}% did NOT pay minimum (serious delinquency)")

RESILIENCE CAPACITY

--- The $400 Emergency Test (pay_casheqv) ---

  Yes..........................................  8,106 ( 65.9%)
  No...........................................  4,189 ( 34.1%)

  💰 65.9% can handle a $400 emergency with cash/equivalent
  ⚠️  34.1% CANNOT handle a $400 emergency

--- Emergency Savings: Months of Expenses Covered (EF2) ---

  No...........................................  3,337 ( 64.9%)
  Yes..........................................  1,807 ( 35.1%)

--- Credit Card Debt Behavior (C3P) ---

  paid at least the minimum payment on all credit cards..  91.4%
  did not use any of my credit cards so had no balances..   6.1%
  did not pay or paid less than the minimum payment on at least one card   2.5%

  → 91.4% paid at least minimum (likely carrying balances)
  → 2.5% did NOT pay minimum (serious delinquency)


## 2.2 Active Shocks: Inflation Responses

In [ ]:
# Visualization 1: The $400 Emergency Test

fig, ax = plt.subplots(figsize=(8, 8))

if 'pay_casheqv' in df.columns:
    cash_data = df['pay_casheqv'].value_counts()
    
    values = [cash_data.get('Yes', 0), cash_data.get('No', 0)]
    labels = ['Yes', 'No']
    colors = ['#2ca02c', '#d62728']
    
    wedges, texts, autotexts = ax.pie(values, labels=labels, autopct='%1.1f%%', 
                                        startangle=90, colors=colors,
                                        textprops={'fontsize': 12, 'fontweight': 'bold'},
                                        wedgeprops=dict(width=0.4, edgecolor='black', linewidth=2),
                                        pctdistance=0.7)
    
    ax.set_title('Can Handle $400 Emergency with Cash/Equivalent', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Emergency Savings - Months of Expenses

fig, ax = plt.subplots(figsize=(12, 6))

if 'EF2' in df.columns:
    ef2_data = df['EF2'].value_counts(normalize=True) * 100
    
    # Sort categories logically
    categories = ef2_data.index.tolist()
    values = [ef2_data.get(cat, 0) for cat in categories]
    
    # Color code by adequacy
    colors_savings = []
    for cat in categories:
        if 'Would not' in str(cat) or 'Less than 1' in str(cat):
            colors_savings.append('#d62728')  # Red - inadequate
        elif '1 to 2' in str(cat) or '3 to 5' in str(cat):
            colors_savings.append('#ff7f0e')  # Orange - moderate
        else:
            colors_savings.append('#2ca02c')  # Green - adequate
    
    bars = ax.barh(range(len(categories)), values, color=colors_savings, 
                    edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(categories)))
    ax.set_yticklabels(categories, fontsize=10)
    ax.set_xlabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_title('Emergency Savings: Months of Expenses Covered', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Credit Card Debt Status

fig, ax = plt.subplots(figsize=(10, 6))

if 'C3P' in df.columns:
    c3p_data = df['C3P'].value_counts(normalize=True) * 100
    
    # Simplify labels
    label_map = {
        'paid at least the minimum payment on all credit cards': 'Paid at least minimum\n(Carrying balance)',
        'did not use any of my credit cards so had no balances': 'No balance',
        'did not pay or paid less than the minimum payment on at least one card': 'Delinquent\n(Less than minimum)'
    }
    
    categories = []
    values = []
    colors = []
    
    for orig_label, short_label in label_map.items():
        if orig_label in c3p_data.index:
            categories.append(short_label)
            values.append(c3p_data[orig_label])
            
            if 'minimum' in short_label and 'Carrying' in short_label:
                colors.append('#ff7f0e')
            elif 'No balance' in short_label:
                colors.append('#2ca02c')
            else:
                colors.append('#d62728')
    
    bars = ax.bar(categories, values, color=colors, edgecolor='black', linewidth=2, alpha=0.85)
    ax.set_ylabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_title('Credit Card Debt Status', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Resilience Capacity

fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

# Chart 1: $400 Emergency Test (Donut Chart)
if 'pay_casheqv' in df.columns:
    cash_data = df['pay_casheqv'].value_counts()
    
    values = [cash_data.get('Yes', 0), cash_data.get('No', 0)]
    labels = ['Can Handle\n$400', 'Cannot Handle\n$400']
    colors = ['#2ca02c', '#d62728']
    
    wedges, texts, autotexts = ax1.pie(values, labels=labels, autopct='%1.1f%%', 
                                        startangle=90, colors=colors,
                                        textprops={'fontsize': 11, 'fontweight': 'bold'},
                                        wedgeprops=dict(width=0.5, edgecolor='black', linewidth=2))
    ax1.set_title('The $400 Emergency Test', fontsize=12, fontweight='bold')

# Chart 2: Emergency Savings Months (Stacked Bar)
if 'EF2' in df.columns:
    ef2_data = df['EF2'].value_counts(normalize=True) * 100
    
    # Sort categories logically
    categories = ef2_data.index.tolist()
    values = [ef2_data.get(cat, 0) for cat in categories]
    
    # Color code by adequacy
    colors_savings = []
    for cat in categories:
        if 'Would not' in str(cat) or 'Less than 1' in str(cat):
            colors_savings.append('#d62728')  # Red - inadequate
        elif '1 to 2' in str(cat) or '3 to 5' in str(cat):
            colors_savings.append('#ff7f0e')  # Orange - moderate
        else:
            colors_savings.append('#2ca02c')  # Green - adequate
    
    bars = ax2.barh(range(len(categories)), values, color=colors_savings, 
                    edgecolor='black', linewidth=1)
    ax2.set_yticks(range(len(categories)))
    ax2.set_yticklabels(categories, fontsize=9)
    ax2.set_xlabel('% of Households', fontsize=11, fontweight='bold')
    ax2.set_title('Emergency Savings:\nMonths of Expenses Covered', fontsize=12, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

# Chart 3: Credit Card Debt (Simple comparison)
if 'C3P' in df.columns:
    c3p_data = df['C3P'].value_counts()
    
    paid_min = c3p_data.get('paid at least the minimum payment on all credit cards', 0)
    no_balance = c3p_data.get('did not use any of my credit cards so had no balances', 0)
    delinquent = c3p_data.get('did not pay or paid less than the minimum payment on at least one card', 0)
    
    categories = ['Paid Min\n(Carrying Balance)', 'No Balance', 'Delinquent\n(< Min Payment)']
    values = [paid_min, no_balance, delinquent]
    colors = ['#ff7f0e', '#2ca02c', '#d62728']
    
    bars = ax3.bar(categories, values, color=colors, edgecolor='black', linewidth=2, alpha=0.8)
    ax3.set_ylabel('Number of Households', fontsize=11, fontweight='bold')
    ax3.set_title('Credit Card Debt Status', fontsize=12, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # Add value labels
    total = sum(values)
    for bar, val in zip(bars, values):
        height = bar.get_height()
        pct = val / total * 100
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(val):,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualization for Inflation Shock Responses

fig, ax = plt.subplots(figsize=(12, 6))

inflation_actions = {
    'INF3_d': 'Reduced spending on\nnon-essentials',
    'INF3_c': 'Reduced spending on\nessentials',
    'INF3_a': 'Used savings',
    'INF3_b': 'Borrowed money or\nused credit cards',
    'INF3_e': 'Worked more hours or\ntook additional job',
    'INF3_f': 'Asked for raise or\nchanged jobs'
}

action_pcts = []
action_labels = []
for var, label in inflation_actions.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        action_pcts.append(pct)
        action_labels.append(label)

# Color code by severity
colors_inflation = []
for var in inflation_actions.keys():
    if var in ['INF3_c', 'INF3_b']:  # Cutting essentials, borrowing - severe
        colors_inflation.append('#d62728')
    elif var in ['INF3_a', 'INF3_d']:  # Using savings, cutting non-essentials - moderate
        colors_inflation.append('#ff7f0e')
    else:  # Working more, asking for raise - proactive
        colors_inflation.append('#2ca02c')

bars = ax.barh(action_labels, action_pcts, color=colors_inflation, 
               edgecolor='black', linewidth=1.5, alpha=0.85)

ax.set_xlabel('Percentage of Households Taking Action', fontsize=12, fontweight='bold')
ax.set_title('How Households Responded to Price Increases', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (bar, pct) in enumerate(zip(bars, action_pcts)):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [9]:
print("="*70)
print("BREAKING POINTS")
print("="*70)

# Healthcare rationing
print("\n--- Healthcare Rationing: Went Without Due to Cost (E1 series) ---\n")

healthcare_barriers = {
    'E1_a': 'Prescription medicine',
    'E1_b': 'See a doctor or specialist',
    'E1_d': 'Dental care'
}

for var, label in healthcare_barriers.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = yes_count / total * 100
            print(f"  {label:.<45} {pct:>5.1f}%")

# Food insecurity
print("\n--- Food Security (FD3) ---\n")
if 'FD3' in df.columns:
    fd3_counts = df['FD3'].value_counts()
    fd3_pct = df['FD3'].value_counts(normalize=True) * 100
    
    for category in fd3_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {fd3_pct[category]:>5.1f}%")
    
    food_insecure = 100 - fd3_pct.get('Enough of the kinds of food we wanted to eat', 0)
    print(f"\n  ⚠️  {food_insecure:.1f}% experienced some level of food insecurity")

# Student loan delinquency
print("\n--- Student Loan Delinquency (SL6) ---\n")
if 'SL6' in df.columns:
    sl6_counts = df['SL6'].value_counts()
    sl6_pct = df['SL6'].value_counts(normalize=True) * 100
    
    for category in sl6_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {sl6_pct[category]:>5.1f}%")

# Banking exclusion
print("\n--- Banking Access (BK1) ---\n")
if 'BK1' in df.columns:
    has_account = (df['BK1'] == 'Yes').sum()
    total = df['BK1'].notna().sum()
    if total > 0:
        pct_banked = has_account / total * 100
        pct_unbanked = 100 - pct_banked
        print(f"  Has checking/savings/money market account.. {pct_banked:.1f}%")
        print(f"  UNBANKED (no account)...................... {pct_unbanked:.1f}%")

print("\n" + "="*70)
print("END OF ACT 2")
print("="*70)

BREAKING POINTS

--- Healthcare Rationing: Went Without Due to Cost (E1 series) ---

  Prescription medicine........................  10.1%
  See a doctor or specialist...................  15.0%
  Dental care..................................  18.8%

--- Food Security (FD3) ---

  Enough of the kinds of food we wanted to eat.  68.0%
  Enough, but not always the kinds of food we wanted to eat  25.6%
  Sometimes not enough to eat..................   4.5%
  Often not enough to eat......................   1.9%

  ⚠️  32.0% experienced some level of food insecurity

--- Student Loan Delinquency (SL6) ---

  No...........................................  80.2%
  Yes..........................................  19.8%

--- Banking Access (BK1) ---

  Has checking/savings/money market account.. 94.7%
  UNBANKED (no account)...................... 5.3%

END OF ACT 2


In [ ]:
# Visualization for Inflation Shock Responses

fig, ax = plt.subplots(figsize=(12, 6))

inflation_actions = {
    'INF3_d': 'Reduced spending on\nnon-essentials',
    'INF3_c': 'Reduced spending on\nessentials',
    'INF3_a': 'Used savings',
    'INF3_b': 'Borrowed money or\nused credit cards',
    'INF3_e': 'Worked more hours or\ntook additional job',
    'INF3_f': 'Asked for raise or\nchanged jobs'
}

action_pcts = []
action_labels = []
for var, label in inflation_actions.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        action_pcts.append(pct)
        action_labels.append(label)

# Color code by severity
colors_inflation = []
for var in inflation_actions.keys():
    if var in ['INF3_c', 'INF3_b']:  # Cutting essentials, borrowing - severe
        colors_inflation.append('#d62728')
    elif var in ['INF3_a', 'INF3_d']:  # Using savings, cutting non-essentials - moderate
        colors_inflation.append('#ff7f0e')
    else:  # Working more, asking for raise - proactive
        colors_inflation.append('#2ca02c')

bars = ax.barh(action_labels, action_pcts, color=colors_inflation, 
               edgecolor='black', linewidth=1.5, alpha=0.85)

ax.set_xlabel('% of Households Taking Action', fontsize=12, fontweight='bold')
ax.set_title('How Households Responded to Price Increases\n(Respondents could select multiple actions)',
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (bar, pct) in enumerate(zip(bars, action_pcts)):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

# Add legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ca02c', edgecolor='black', label='Proactive (Work/Income)'),
    Patch(facecolor='#ff7f0e', edgecolor='black', label='Moderate (Savings/Non-essentials)'),
    Patch(facecolor='#d62728', edgecolor='black', label='Severe (Essentials/Borrowing)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

# Visualization for Breaking Points - Healthcare Rationing

fig, ax = plt.subplots(figsize=(10, 6))

healthcare_barriers = {
    'E1_d': 'Dental care',
    'E1_b': 'See doctor/specialist',
    'E1_a': 'Prescription medicine',
    'E1_c': 'Mental health care'
}

health_pcts = []
health_labels = []
for var, label in healthcare_barriers.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        health_pcts.append(pct)
        health_labels.append(label)

bars = ax.barh(health_labels, health_pcts, color='#d62728', 
               edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Percentage Who Went Without Due to Cost', fontsize=12, fontweight='bold')
ax.set_title('Healthcare Rationing', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, health_pcts)):
    ax.text(pct + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Breaking Points - Food Security

fig, ax = plt.subplots(figsize=(8, 8))

if 'FD3' in df.columns:
    fd3_data = df['FD3'].value_counts(normalize=True) * 100
    
    labels_food = []
    values_food = []
    colors_food = []
    
    for cat in fd3_data.index:
        if pd.notna(cat):
            labels_food.append(cat)
            values_food.append(fd3_data[cat])
            
            # Color code
            if 'Enough of the kinds' in str(cat):
                colors_food.append('#2ca02c')  # Green - secure
            elif 'Enough but not' in str(cat):
                colors_food.append('#ff7f0e')  # Orange - marginal
            else:
                colors_food.append('#d62728')  # Red - insecure
    
    wedges, texts, autotexts = ax.pie(values_food, labels=labels_food, autopct='%1.1f%%',
                                        startangle=90, colors=colors_food,
                                        textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_title('Food Security Status', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Breaking Points - Banking and Student Loans

fig, ax = plt.subplots(figsize=(10, 6))

exclusion_data = []
exclusion_labels = []
exclusion_colors = []

# Banking exclusion
if 'BK1' in df.columns:
    unbanked_pct = ((df['BK1'] != 'Yes').sum() / df['BK1'].notna().sum()) * 100
    exclusion_data.append(unbanked_pct)
    exclusion_labels.append('Unbanked\n(No account)')
    exclusion_colors.append('#d62728')

# Student loan delinquency (of those with loans)
if 'SL6' in df.columns:
    behind = ((df['SL6'] == 'Yes').sum() / df['SL6'].notna().sum()) * 100
    exclusion_data.append(behind)
    exclusion_labels.append('Behind on\nstudent loans')
    exclusion_colors.append('#ff7f0e')

bars = ax.bar(exclusion_labels, exclusion_data, color=exclusion_colors,
              edgecolor='black', linewidth=2, alpha=0.8, width=0.5)
ax.set_ylabel('Percentage', fontsize=12, fontweight='bold')
ax.set_title('Financial Exclusion Indicators', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, exclusion_data):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3.1 Directional Momentum

In [ ]:
# Visualization for Breaking Points

fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# Chart 1: Healthcare Rationing (Top left)
ax1 = fig.add_subplot(gs[0, 0])

healthcare_barriers = {
    'E1_d': 'Dental care',
    'E1_b': 'See doctor/specialist',
    'E1_a': 'Prescription medicine',
    'E1_c': 'Mental health care'
}

health_pcts = []
health_labels = []
for var, label in healthcare_barriers.items():
    if var in df.columns:
        pct = (df[var] == 'Yes').sum() / df[var].notna().sum() * 100
        health_pcts.append(pct)
        health_labels.append(label)

bars1 = ax1.barh(health_labels, health_pcts, color='#d62728', 
                 edgecolor='black', linewidth=1.5, alpha=0.8)
ax1.set_xlabel('% Who Went Without Due to Cost', fontsize=11, fontweight='bold')
ax1.set_title('Healthcare Rationing', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars1, health_pcts)):
    ax1.text(pct + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=10, fontweight='bold')

# Chart 2: Food Security (Top right)
ax2 = fig.add_subplot(gs[0, 1])

if 'FD3' in df.columns:
    fd3_data = df['FD3'].value_counts(normalize=True) * 100
    
    labels_food = []
    values_food = []
    colors_food = []
    
    for cat in fd3_data.index:
        if pd.notna(cat):
            labels_food.append(cat)
            values_food.append(fd3_data[cat])
            
            # Color code
            if 'Enough of the kinds' in str(cat):
                colors_food.append('#2ca02c')  # Green - secure
            elif 'Enough but not' in str(cat):
                colors_food.append('#ff7f0e')  # Orange - marginal
            else:
                colors_food.append('#d62728')  # Red - insecure
    
    wedges, texts, autotexts = ax2.pie(values_food, labels=labels_food, autopct='%1.1f%%',
                                        startangle=90, colors=colors_food,
                                        textprops={'fontsize': 9, 'fontweight': 'bold'})
    ax2.set_title('Food Security Status', fontsize=12, fontweight='bold')

# Chart 3: Banking Access & Student Loans (Bottom left)
ax3 = fig.add_subplot(gs[1, 0])

exclusion_data = []
exclusion_labels = []
exclusion_colors = []

# Banking exclusion
if 'BK1' in df.columns:
    unbanked_pct = ((df['BK1'] != 'Yes').sum() / df['BK1'].notna().sum()) * 100
    exclusion_data.append(unbanked_pct)
    exclusion_labels.append('Unbanked\n(No account)')
    exclusion_colors.append('#d62728')

# Student loan delinquency (of those with loans)
if 'SL6' in df.columns:
    behind = ((df['SL6'] == 'Yes').sum() / df['SL6'].notna().sum()) * 100
    exclusion_data.append(behind)
    exclusion_labels.append('Behind on\nstudent loans')
    exclusion_colors.append('#ff7f0e')

bars3 = ax3.bar(exclusion_labels, exclusion_data, color=exclusion_colors,
                edgecolor='black', linewidth=2, alpha=0.8, width=0.6)
ax3.set_ylabel('Percentage', fontsize=11, fontweight='bold')
ax3.set_title('Financial Exclusion Indicators', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val in zip(bars3, exclusion_data):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 4: Summary heatmap of hardship (Bottom right)
ax4 = fig.add_subplot(gs[1, 1])

# Create summary of breaking points
hardship_summary = {
    'Dental care rationing': health_pcts[0] if len(health_pcts) > 0 else 0,
    'Doctor/specialist rationing': health_pcts[1] if len(health_pcts) > 1 else 0,
    'Prescription rationing': health_pcts[2] if len(health_pcts) > 2 else 0,
    'Food insecurity': 100 - fd3_data.get('Enough of the kinds of food we wanted to eat', 0) if 'FD3' in df.columns else 0,
    'Unbanked': exclusion_data[0] if len(exclusion_data) > 0 else 0,
    'Student loan delinquency': exclusion_data[1] if len(exclusion_data) > 1 else 0
}

categories_heat = list(hardship_summary.keys())
values_heat = list(hardship_summary.values())

# Create color gradient based on severity
colors_heat = []
for val in values_heat:
    if val < 10:
        colors_heat.append('#2ca02c')
    elif val < 25:
        colors_heat.append('#ff7f0e')
    else:
        colors_heat.append('#d62728')

bars4 = ax4.barh(categories_heat, values_heat, color=colors_heat,
                 edgecolor='black', linewidth=1.5, alpha=0.85)
ax4.set_xlabel('% Experiencing Hardship', fontsize=11, fontweight='bold')
ax4.set_title('Breaking Points Summary', fontsize=12, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars4, values_heat)):
    ax4.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Directional Momentum

fig, ax = plt.subplots(figsize=(10, 6))

if 'B3' in df.columns:
    b3_data = df['B3'].value_counts(normalize=True) * 100
    
    # Group into better/same/worse
    better_cats = ['Much better off', 'Somewhat better off']
    same_cats = ['About the same']
    worse_cats = ['Somewhat worse off', 'Much worse off']
    
    better = sum([b3_data.get(cat, 0) for cat in better_cats])
    same = sum([b3_data.get(cat, 0) for cat in same_cats])
    worse = sum([b3_data.get(cat, 0) for cat in worse_cats])
    
    # Create bar chart
    categories = ['Better Off', 'About the Same', 'Worse Off']
    values = [better, same, worse]
    colors = ['#2ca02c', '#1f77b4', '#d62728']
    
    bars = ax.barh(categories, values, color=colors, edgecolor='black', linewidth=2, alpha=0.85)
    
    ax.set_xlabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_title('Financial Situation vs. 12 Months Ago', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3.2 Future Preparedness: Retirement

In [11]:
print("="*70)
print("RETIREMENT PREPAREDNESS")
print("="*70)

print("\n--- Retirement Savings on Track (K0) ---\n")
if 'K0' in df.columns:
    k0_counts = df['K0'].value_counts()
    k0_pct = df['K0'].value_counts(normalize=True) * 100
    
    for category in k0_counts.index:
        if pd.notna(category):
            print(f"  {category:.<45} {k0_pct[category]:>5.1f}%")
    
    on_track = k0_pct.get('Yes', 0)
    print(f"\n  → {on_track:.1f}% feel their retirement savings are on track")

# Has any retirement savings
print("\n--- Has Retirement Savings (K21 series) ---\n")
retirement_types = ['K21_a', 'K21_b', 'K21_c', 'K21_d', 'K21_e', 'K21_f']
has_any_retirement = df[retirement_types].apply(lambda row: (row == 'Yes').any(), axis=1).sum()
total_respondents = df[retirement_types].notna().any(axis=1).sum()

if total_respondents > 0:
    pct_has_retirement = has_any_retirement / total_respondents * 100
    print(f"  Has any retirement savings/assets.......... {pct_has_retirement:.1f}%")
    print(f"  Has NO retirement savings/assets........... {100-pct_has_retirement:.1f}%")

RETIREMENT PREPAREDNESS

--- Retirement Savings on Track (K0) ---

  No...........................................  43.7%
  Yes..........................................  37.7%
  Don’t know...................................  18.6%

  → 37.7% feel their retirement savings are on track

--- Has Retirement Savings (K21 series) ---

  Has any retirement savings/assets.......... 82.1%
  Has NO retirement savings/assets........... 17.9%


In [ ]:
# Visualization for Retirement - On Track Status

fig, ax = plt.subplots(figsize=(8, 8))

if 'K0' in df.columns:
    k0_data = df['K0'].value_counts()
    
    values = []
    labels = []
    for cat in k0_data.index:
        if pd.notna(cat):
            values.append(k0_data[cat])
            labels.append(cat)
    
    colors = ['#2ca02c', '#d62728', '#ff7f0e']
    
    wedges, texts, autotexts = ax.pie(values, labels=labels, autopct='%1.1f%%',
                                        startangle=90, colors=colors,
                                        textprops={'fontsize': 11, 'fontweight': 'bold'},
                                        wedgeprops=dict(width=0.4, edgecolor='black', linewidth=2),
                                        pctdistance=0.7)
    ax.set_title('Retirement Savings On Track?', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Retirement - Savings Ownership

fig, ax = plt.subplots(figsize=(10, 6))

retirement_types = ['K21_a', 'K21_b', 'K21_c', 'K21_d', 'K21_e', 'K21_f']
has_any = df[retirement_types].apply(lambda row: (row == 'Yes').any(), axis=1).sum()
total_respondents = df[retirement_types].notna().any(axis=1).sum()
no_savings = total_respondents - has_any

categories = ['Has Retirement Savings/Assets', 'No Retirement Savings']
values = [has_any, no_savings]
percentages = [val / total_respondents * 100 for val in values]
colors = ['#2ca02c', '#d62728']

bars = ax.bar(categories, percentages, color=colors, edgecolor='black', linewidth=2, alpha=0.85)
ax.set_ylabel('Percentage of Households', fontsize=12, fontweight='bold')
ax.set_title('Retirement Savings Ownership', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add labels
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{pct:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3.3 Overall Financial Health Status

In [ ]:
# Visualization for Overall Financial Situation - B2 Distribution

fig, ax = plt.subplots(figsize=(12, 3))

if 'B2' in df.columns:
    b2_data = df['B2'].value_counts(normalize=True) * 100
    
    order = ['Living comfortably', 'Doing okay', 'Just getting by', 'Finding it difficult to get by']
    values = [b2_data.get(cat, 0) for cat in order]
    
    # Create stacked 100% bar
    colors_b2 = ['#2ca02c', '#90ee90', '#ff7f0e', '#d62728']
    
    left = 0
    for i, (val, color, cat) in enumerate(zip(values, colors_b2, order)):
        ax.barh(0, val, left=left, color=color, edgecolor='black', linewidth=1.5, height=0.5)
        
        # Add label
        if val > 5:  # Only show label if segment is large enough
            ax.text(left + val/2, 0, f'{val:.1f}%\n{cat}', 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        left += val
    
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_yticks([])
    ax.set_title('Overall Financial Situation Distribution', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Financial Health Tiers

fig, ax = plt.subplots(figsize=(10, 6))

if 'B2' in df.columns:
    b2_data = df['B2'].value_counts(normalize=True) * 100
    
    # Use actual B2 responses as labels (no Thriving/Stable/Vulnerable)
    order = ['Living comfortably', 'Doing okay', 'Just getting by', 'Finding it difficult to get by']
    values = [b2_data.get(cat, 0) for cat in order]
    colors_tiers = ['#2ca02c', '#90ee90', '#ff7f0e', '#d62728']
    
    bars = ax.barh(order, values, color=colors_tiers, 
                    edgecolor='black', linewidth=2, alpha=0.85)
    
    ax.set_xlabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax.set_title('Financial Health by Status', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for bar, val in zip(bars, values):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Retirement Preparedness

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Retirement on Track (Donut)
if 'K0' in df.columns:
    k0_data = df['K0'].value_counts()
    
    values = []
    labels = []
    for cat in k0_data.index:
        if pd.notna(cat):
            values.append(k0_data[cat])
            labels.append(cat)
    
    colors = ['#2ca02c', '#d62728', '#ff7f0e']
    
    wedges, texts, autotexts = ax1.pie(values, labels=labels, autopct='%1.1f%%',
                                        startangle=90, colors=colors,
                                        textprops={'fontsize': 11, 'fontweight': 'bold'},
                                        wedgeprops=dict(width=0.5, edgecolor='black', linewidth=2))
    ax1.set_title('Retirement Savings On Track?', fontsize=13, fontweight='bold')

# Chart 2: Has Retirement Savings (Binary)
retirement_types = ['K21_a', 'K21_b', 'K21_c', 'K21_d', 'K21_e', 'K21_f']
has_any = df[retirement_types].apply(lambda row: (row == 'Yes').any(), axis=1).sum()
no_savings = df[retirement_types].notna().any(axis=1).sum() - has_any

categories = ['Has Retirement\nSavings/Assets', 'No Retirement\nSavings']
values = [has_any, no_savings]
colors = ['#2ca02c', '#d62728']

bars = ax2.bar(categories, values, color=colors, edgecolor='black', linewidth=2, alpha=0.85)
ax2.set_ylabel('Number of Households', fontsize=12, fontweight='bold')
ax2.set_title('Retirement Savings Ownership', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add labels
total = sum(values)
for bar, val in zip(bars, values):
    height = bar.get_height()
    pct = val / total * 100
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val):,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualization for Financial Control

fig, ax = plt.subplots(figsize=(10, 6))

control_vars = {
    'B1_a': 'Control over day-to-day finances',
    'B1_b': 'Financial situation will work out'
}

control_pcts = []
control_labels = []
for var, label in control_vars.items():
    if var in df.columns:
        pct = df[var].isin(['Always', 'Often']).sum() / df[var].notna().sum() * 100
        control_pcts.append(pct)
        control_labels.append(label)

bars = ax.barh(control_labels, control_pcts, color='#2ca02c', 
               edgecolor='black', linewidth=2, alpha=0.85)
ax.set_xlabel('Percentage Who Feel This Way Always/Often', fontsize=12, fontweight='bold')
ax.set_title('Sense of Financial Control', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, control_pcts)):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization for Financial Stress

fig, ax = plt.subplots(figsize=(10, 6))

stress_vars = {
    'B0_a': 'Stressed about personal finances',
    'B0_b': 'Finances control my life',
    'B0_c': 'Worry about running out of money in retirement'
}

stress_pcts = []
stress_labels = []
for var, label in stress_vars.items():
    if var in df.columns:
        pct = df[var].isin(['Completely', 'Very well', 'Somewhat']).sum() / df[var].notna().sum() * 100
        stress_pcts.append(pct)
        stress_labels.append(label)

bars = ax.barh(stress_labels, stress_pcts, color='#d62728', 
               edgecolor='black', linewidth=2, alpha=0.85)
ax.set_xlabel('Percentage Experiencing This Stress', fontsize=12, fontweight='bold')
ax.set_title('Financial Stress Levels', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, stress_pcts)):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [13]:
print("="*70)
print("PSYCHOLOGICAL WELL-BEING")
print("="*70)

# Financial control
print("\n--- Sense of Financial Control (B1 series) ---\n")

control_vars = {
    'B1_a': 'I have control over day-to-day finances',
    'B1_b': 'Financial situation will work out'
}

for var, label in control_vars.items():
    if var in df.columns:
        # B1 uses: Always, Often, Sometimes, Rarely, Never
        positive = df[var].isin(['Always', 'Often']).sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = positive / total * 100
            print(f"  {label}")
            print(f"    Feel this way Always/Often............. {pct:.1f}%\n")

# Financial stress
print("--- Financial Stress (B0 series) ---\n")

stress_vars = {
    'B0_a': 'Stressed about personal finances',
    'B0_b': 'Finances control my life',
    'B0_c': 'Worry about running out of money in retirement'
}

for var, label in stress_vars.items():
    if var in df.columns:
        # B0 uses: Completely, Very well, Somewhat, Very little, Not at all
        stressed = df[var].isin(['Completely', 'Very well', 'Somewhat']).sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = stressed / total * 100
            print(f"  {label}")
            print(f"    Feel this way (Completely/Very well/Somewhat) {pct:.1f}%\n")

print("\n" + "="*70)
print("END OF ACT 3")
print("="*70)

PSYCHOLOGICAL WELL-BEING

--- Sense of Financial Control (B1 series) ---

  I have control over day-to-day finances
    Feel this way Always/Often............. 42.8%

  Financial situation will work out
    Feel this way Always/Often............. 25.2%

--- Financial Stress (B0 series) ---

  Stressed about personal finances
    Feel this way (Completely/Very well/Somewhat) 47.7%

  Finances control my life
    Feel this way (Completely/Very well/Somewhat) 54.7%

  Worry about running out of money in retirement
    Feel this way (Completely/Very well/Somewhat) 63.4%


END OF ACT 3


In [ ]:
# Visualization for Overall Financial Health Status

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Chart 1: B2 Distribution (Stacked Horizontal)
if 'B2' in df.columns:
    b2_data = df['B2'].value_counts(normalize=True) * 100
    
    order = ['Living comfortably', 'Doing okay', 'Just getting by', 'Finding it difficult to get by']
    values = [b2_data.get(cat, 0) for cat in order]
    
    # Create stacked 100% bar
    colors_b2 = ['#2ca02c', '#90ee90', '#ff7f0e', '#d62728']
    
    left = 0
    for i, (val, color, cat) in enumerate(zip(values, colors_b2, order)):
        ax1.barh(0, val, left=left, color=color, edgecolor='black', linewidth=1.5, height=0.5)
        
        # Add label
        if val > 5:  # Only show label if segment is large enough
            ax1.text(left + val/2, 0, f'{val:.1f}%\n{cat}', 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        left += val
    
    ax1.set_xlim(0, 100)
    ax1.set_ylim(-0.5, 0.5)
    ax1.set_xlabel('Percentage of Households', fontsize=12, fontweight='bold')
    ax1.set_yticks([])
    ax1.set_title('Overall Financial Situation Distribution', fontsize=13, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

# Chart 2: Three-Tier Health Status
if 'B2' in df.columns:
    thriving = b2_data.get('Living comfortably', 0)
    stable = b2_data.get('Doing okay', 0)
    vulnerable = b2_data.get('Just getting by', 0) + b2_data.get('Finding it difficult to get by', 0)
    
    tiers = ['🟢 Thriving\n(Living comfortably)', 
             '🟡 Stable\n(Doing okay)', 
             '🔴 Vulnerable\n(Just getting by/difficult)']
    tier_values = [thriving, stable, vulnerable]
    tier_colors = ['#2ca02c', '#ffd700', '#d62728']
    
    # Create pyramid/funnel chart
    bars = ax2.barh(tiers, tier_values, color=tier_colors, 
                    edgecolor='black', linewidth=2, alpha=0.85)
    
    ax2.set_xlabel('% of Households', fontsize=12, fontweight='bold')
    ax2.set_title('Financial Health Tiers', fontsize=13, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for bar, val in zip(bars, tier_values):
        ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
---

# Summary

This three-act analysis reveals the complete picture of American household financial health in 2024:

**ACT 1** showed us the foundation—income sources, employment status, housing costs, and the critical gap between income and spending.

**ACT 2** stress-tested that foundation—examining emergency savings, inflation responses, and the breaking points where households sacrifice healthcare, food security, and basic financial access.

**ACT 3** revealed the trajectory—whether households are improving or declining, their retirement preparedness, and their psychological relationship with money.

Together, these metrics paint a comprehensive portrait of financial resilience, vulnerability, and well-being across American households.

In [ ]:
# Visualization for Psychological Well-being

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Chart 1: Financial Control (Positive indicators)
control_vars = {
    'B1_a': 'Control over day-to-day\nfinances',
    'B1_b': 'Financial situation\nwill work out'
}

control_pcts = []
control_labels = []
for var, label in control_vars.items():
    if var in df.columns:
        pct = df[var].isin(['Always', 'Often']).sum() / df[var].notna().sum() * 100
        control_pcts.append(pct)
        control_labels.append(label)

bars1 = ax1.barh(control_labels, control_pcts, color='#2ca02c', 
                 edgecolor='black', linewidth=2, alpha=0.85)
ax1.set_xlabel('% Who Feel This Way Always/Often', fontsize=11, fontweight='bold')
ax1.set_title('Sense of Financial Control\n(Positive Indicators)', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars1, control_pcts)):
    ax1.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

# Chart 2: Financial Stress (Negative indicators)
stress_vars = {
    'B0_a': 'Stressed about\npersonal finances',
    'B0_b': 'Finances control\nmy life',
    'B0_c': 'Worry about running out\nof money in retirement'
}

stress_pcts = []
stress_labels = []
for var, label in stress_vars.items():
    if var in df.columns:
        pct = df[var].isin(['Completely', 'Very well', 'Somewhat']).sum() / df[var].notna().sum() * 100
        stress_pcts.append(pct)
        stress_labels.append(label)

bars2 = ax2.barh(stress_labels, stress_pcts, color='#d62728', 
                 edgecolor='black', linewidth=2, alpha=0.85)
ax2.set_xlabel('% Experiencing This Stress', fontsize=11, fontweight='bold')
ax2.set_title('Financial Stress Levels\n(Negative Indicators)', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars2, stress_pcts)):
    ax2.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()